In [ ]:

import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from itertools import product
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score
from sklearn.svm import OneClassSVM
from pyod.models.knn import KNN
from sklearn.model_selection import train_test_split
pd.set_option('display.max_columns', None)

## DATA PREP

In [2]:
df = pd.read_csv('../data_cleaned.csv', low_memory=False)

In [3]:
# All columns to use in model 
continuous_features = [
        'FIT101', 'LIT101',
        'AIT201', 'AIT202', 'AIT203', 'FIT201',
        'AIT301', 'AIT302', 'AIT303', 'DPIT301', 'FIT301', 'LIT301',
        'AIT401', 'AIT402', 'FIT401', 'LIT401',
        'AIT501', 'AIT502', 'AIT503', 'AIT504',
        'FIT501', 'FIT502', 'FIT503', 'FIT504',
        'PIT501', 'PIT502', 'PIT503',
        'FIT601', 'FIT602', 'LIT601', 'LIT602'
    ]
    
binary_features = [
        'MV101', 'P101', 'P102',
        'MV201', 'P201', 'P202', 'P203', 'P204', 'P205', 'P206',
        'MV301', 'MV302', 'MV303', 'MV304', 'P301', 'P302',
        'P401', 'P402', 'P403', 'P404', 'UV401',
        'MV501', 'MV502', 'MV503', 'MV504', 'P501', 'P502',
        'P601', 'P602','P603']

all_columns_to_use = continuous_features+binary_features

In [ ]:
def create_sliding_windows_matrix(df, feature_cols, stride, window_size=60, label_mode="any"):
    """
    Converts the dataset into overlapping windows.

    Args:
        df: DataFrame with features and 'Anomaly' column.
        feature_cols: list of features to use.
        window_size: number of timesteps per window.
        stride: number of steps to move between windows.
        label_mode: 'any' (1 if any anomaly in window)
                    or 'majority' (1 if >50% anomalies)
    Returns:
        X_windows: np.array [n_windows, window_size * n_features]
        y_windows: np.array [n_windows]
    """
    X_windows, y_windows = [], []
    values = df[feature_cols].values
    labels = df["Anomaly"].values

    for start in range(0, len(df) - window_size + 1, stride):
        end = start + window_size
        window = values[start:end]
        label_window = labels[start:end]

        if label_mode == "any":
            label = 1 if np.any(label_window > 0) else 0
        elif label_mode == "majority":
            label = 1 if np.mean(label_window) > 0.5 else 0
        else:
            raise ValueError("label_mode must be 'any' or 'majority'")

        X_windows.append(window.flatten())
        y_windows.append(label)

    return np.array(X_windows, dtype=np.float32), np.array(y_windows, dtype=np.int8)

selected_features = all_columns_to_use
X_all, y_all = create_sliding_windows_matrix(df, selected_features, window_size=60, stride=10, label_mode="majority")

X_train = X_all[y_all == 0]
X_test = X_all
y_test = y_all

print(f"Train windows: {len(X_train)} | Test windows: {len(X_test)}")

# Standardization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


use_pca = True  

if use_pca:
    pca = PCA(n_components=0.95, random_state=42)
    X_train_scaled = pca.fit_transform(X_train_scaled)
    X_test_scaled = pca.transform(X_test_scaled)
    print(f"PCA applied: reduced to {X_train_scaled.shape[1]} components")


Train windows: 9952 | Test windows: 11611
PCA applied: reduced to 47 components


## Isolation Forest

In [5]:

# ISolation forest

param_grid = {
    "contamination": [0.001, 0.005, 0.01, 0.02],
    "max_samples": [128, 256, 512],
    "max_features": [0.5, 0.7, 1.0],
    "n_estimators": [200, 400, 800]
}

best_f1 = 0
best_model = None
best_params = {}

for contamination, max_samples, max_features, n_estimators in product(
        param_grid["contamination"],
        param_grid["max_samples"],
        param_grid["max_features"],
        param_grid["n_estimators"]
):
    iso = IsolationForest(
        n_estimators=n_estimators,
        max_samples=max_samples,
        contamination=contamination,
        max_features=max_features,
        random_state=42,
        n_jobs=-1
    )
    iso.fit(X_train_scaled)
    y_pred = (iso.predict(X_test_scaled) == -1).astype(int)

    report = classification_report(y_test, y_pred, digits=4, output_dict=True)
    f1 = report['1']['f1-score']

    if f1 > best_f1:
        best_f1 = f1
        best_model = iso
        best_params = {
            "contamination": contamination,
            "max_samples": max_samples,
            "max_features": max_features,
            "n_estimators": n_estimators
        }
        print(f"New best F1={f1:.4f} | Params={best_params}")


y_pred_final = (best_model.predict(X_test_scaled) == -1).astype(int)

report = classification_report(y_test, y_pred_final, digits=4)
roc_auc = roc_auc_score(y_test, y_pred_final)
pr_auc = average_precision_score(y_test, y_pred_final)

print("\n Best parameters:", best_params)
print("\n Final Isolation Forest (improved benchmark):")
print(report)
print(f"ROC-AUC: {roc_auc:.4f}")
print(f"PR-AUC:  {pr_auc:.4f}")


New best F1=0.0024 | Params={'contamination': 0.001, 'max_samples': 128, 'max_features': 0.5, 'n_estimators': 200}
New best F1=0.0060 | Params={'contamination': 0.001, 'max_samples': 128, 'max_features': 0.5, 'n_estimators': 400}
New best F1=0.0231 | Params={'contamination': 0.005, 'max_samples': 128, 'max_features': 0.5, 'n_estimators': 200}
New best F1=0.0531 | Params={'contamination': 0.01, 'max_samples': 128, 'max_features': 0.5, 'n_estimators': 200}
New best F1=0.0585 | Params={'contamination': 0.01, 'max_samples': 128, 'max_features': 0.5, 'n_estimators': 400}
New best F1=0.0617 | Params={'contamination': 0.01, 'max_samples': 128, 'max_features': 0.7, 'n_estimators': 400}
New best F1=0.0702 | Params={'contamination': 0.01, 'max_samples': 128, 'max_features': 0.7, 'n_estimators': 800}
New best F1=0.0776 | Params={'contamination': 0.01, 'max_samples': 256, 'max_features': 0.5, 'n_estimators': 200}
New best F1=0.0807 | Params={'contamination': 0.01, 'max_samples': 512, 'max_features

## OCSVM

In [11]:

param_grid_svm = {
    "kernel": ["rbf", "sigmoid"],
    "nu": [0.001, 0.005, 0.01, 0.05],
    "gamma": ["scale", "auto", 0.001, 0.01, 0.1]
}

best_f1_svm = 0
best_model_svm = None
best_params_svm = {}

for kernel, nu, gamma in product(
        param_grid_svm["kernel"],
        param_grid_svm["nu"],
        param_grid_svm["gamma"]
):
    ocsvm = OneClassSVM(kernel=kernel, nu=nu, gamma=gamma)
    ocsvm.fit(X_train_scaled)
    y_pred = (ocsvm.predict(X_test_scaled) == -1).astype(int)

    report = classification_report(y_test, y_pred, digits=4, output_dict=True)
    f1 = report['1']['f1-score']

    if f1 > best_f1_svm:
        best_f1_svm = f1
        best_model_svm = ocsvm
        best_params_svm = {
            "kernel": kernel,
            "nu": nu,
            "gamma": gamma
        }
        print(f" New best OCSVM F1={f1:.4f} | Params={best_params_svm}")

y_pred_svm = (best_model_svm.predict(X_test_scaled) == -1).astype(int)
roc_auc_svm = roc_auc_score(y_test, y_pred_svm)
pr_auc_svm = average_precision_score(y_test, y_pred_svm)
print("\n Best One-Class SVM parameters:", best_params_svm)
print(classification_report(y_test, y_pred_svm, digits=4))
print(f"ROC-AUC: {roc_auc_svm:.4f}")
print(f"PR-AUC:  {pr_auc_svm:.4f}")



 New best OCSVM F1=0.5130 | Params={'kernel': 'rbf', 'nu': 0.001, 'gamma': 'scale'}
 New best OCSVM F1=0.7320 | Params={'kernel': 'rbf', 'nu': 0.001, 'gamma': 0.001}
 New best OCSVM F1=0.7383 | Params={'kernel': 'rbf', 'nu': 0.005, 'gamma': 0.001}
 New best OCSVM F1=0.7486 | Params={'kernel': 'rbf', 'nu': 0.01, 'gamma': 0.001}

 Best One-Class SVM parameters: {'kernel': 'rbf', 'nu': 0.01, 'gamma': 0.001}
              precision    recall  f1-score   support

           0     0.9466    0.9804    0.9632      9952
           1     0.8505    0.6685    0.7486      1659

    accuracy                         0.9358     11611
   macro avg     0.8985    0.8244    0.8559     11611
weighted avg     0.9329    0.9358    0.9326     11611

ROC-AUC: 0.8244
PR-AUC:  0.6159


## KNN

In [14]:


# --- KNN parameter grid ---
param_grid_knn = {
    "n_neighbors": [5, 10, 20],
    "method": ["largest", "mean", "median"],  # how distance is aggregated
    "contamination": [0.005, 0.01, 0.02]
}

best_f1_knn = 0
best_model_knn = None
best_params_knn = {}

for n_neighbors, method, contamination in product(
        param_grid_knn["n_neighbors"],
        param_grid_knn["method"],
        param_grid_knn["contamination"]
):
    knn = KNN(
        n_neighbors=n_neighbors,
        method=method,
        contamination=contamination,
        n_jobs=-1
    )
    knn.fit(X_train_scaled)
    y_pred = knn.predict(X_test_scaled)  # 1 = outlier, 0 = inlier

    report = classification_report(y_test, y_pred, digits=4, output_dict=True)
    f1 = report['1']['f1-score']

    if f1 > best_f1_knn:
        best_f1_knn = f1
        best_model_knn = knn
        best_params_knn = {
            "n_neighbors": n_neighbors,
            "method": method,
            "contamination": contamination
        }
        print(f"🚀 New best KNN F1={f1:.4f} | Params={best_params_knn}")

# --- Final evaluation ---
y_pred_knn = best_model_knn.predict(X_test_scaled)
roc_auc_knn = roc_auc_score(y_test, y_pred_knn)
pr_auc_knn = average_precision_score(y_test, y_pred_knn)

print("\n✅ Best KNN parameters:", best_params_knn)
print(classification_report(y_test, y_pred_knn, digits=4))
print(f"ROC-AUC: {roc_auc_knn:.4f}")
print(f"PR-AUC:  {pr_auc_knn:.4f}")


🚀 New best KNN F1=0.0628 | Params={'n_neighbors': 5, 'method': 'largest', 'contamination': 0.005}
🚀 New best KNN F1=0.2950 | Params={'n_neighbors': 5, 'method': 'largest', 'contamination': 0.01}
🚀 New best KNN F1=0.6100 | Params={'n_neighbors': 5, 'method': 'largest', 'contamination': 0.02}
🚀 New best KNN F1=0.6862 | Params={'n_neighbors': 5, 'method': 'mean', 'contamination': 0.02}

✅ Best KNN parameters: {'n_neighbors': 5, 'method': 'mean', 'contamination': 0.02}
              precision    recall  f1-score   support

           0     0.9313    0.9868    0.9583      9952
           1     0.8771    0.5636    0.6862      1659

    accuracy                         0.9264     11611
   macro avg     0.9042    0.7752    0.8223     11611
weighted avg     0.9236    0.9264    0.9194     11611

ROC-AUC: 0.7752
PR-AUC:  0.5567
